# 04 · Model comparison (Team 10, Deliverable 4)

Score every candidate on the **same** forecast sets, the **same** zero-filled join and the
**same** metrics, at two origins, then apply the carry rule that was written down before any
score existed (`carried_model` in `src/forecasting.py`, and `results/baseline-decision.md`).

| Candidate | Why it is in |
|---|---|
| Seasonal naive | The reference. Same route, same month, year before T, for both years. No settings. |
| Trend-adjusted naive | Tests whether a simple damped growth rate beats "repeat last year". |
| Ridge growth | Linear model of log growth on route, airport and calendar inputs; readable coefficients. |
| Gradient-boosted growth | Trees on the same inputs; captures interactions (e.g. market × month). |

**Not fitted:** network ARIMA/SARIMAX (one series, cannot produce a route plan), per-route ARIMA
(thousands of short, broken series), LSTM (short clean history and a new dependency), ensembles
and hyperparameter searches (the select window overlaps the test window, so nothing may be tuned
on it).

**Origins** (from the data): *select* = test − 12 months, *stability* = test − 24 months.

In [ ]:
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

MODELING = next(p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "team-10-modeling"]
                if (p / "src" / "forecasting.py").exists())
sys.path.insert(0, str(MODELING / "src"))

from data import BANNER, FIGS, TABLES, load_airports, load_pax, tag, write_table
from features import airport_table, clean_pax, route_table
import forecasting as F

print(BANNER)
pax = clean_pax(load_pax())
routes = route_table(pax, airport_table(load_airports()))
o = F.origins(pax)
print({k: str(v.date()) for k, v in o.__dict__.items()})

In [ ]:
results, horizons = {}, {}
for label, T in (("select", o.select), ("stability", o.stability)):
    fs = F.forecast_set(pax, T)
    results[label], horizons[label] = {}, {}
    for name in F.CANDIDATES:
        t0 = time.perf_counter()
        try:
            scored = F.scored_frame(pax, fs, F.fit_predict(name, pax, routes, fs, T), T)
        except F.InsufficientHistory as e:
            results[label][name] = None
            print(f"{label:9s} {name:26s} NOT FITTABLE: {e}")
            continue
        results[label][name] = F.summary(scored)
        horizons[label][name] = F.by_horizon(scored)
        print(f"{label:9s} {name:26s} WAPE y1 {results[label][name]['WAPE_y1']:.3f}  "
              f"y2 {results[label][name]['WAPE_y2']:.3f}  ({time.perf_counter() - t0:.1f}s)")

In [ ]:
table = pd.concat({k: pd.DataFrame({n: r for n, r in v.items() if r is not None}).T
                   for k, v in results.items()}, names=["origin", "model"])
table.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, label in zip(axes, ("select", "stability")):
    for name, s in horizons[label].items():
        ax.plot(s.index, 100 * s.values, marker="o", ms=3, label=name)
    ax.axvline(12.5, color="k", lw=0.8, ls=":")
    ax.set(xlabel="months after origin", title=tag(f"Route-month WAPE by horizon — {label} origin"))
axes[0].set_ylabel("WAPE (%)")
axes[0].legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIGS / "d4_wape_by_horizon.png", dpi=120); plt.show()

## Apply the carry rule

In [ ]:
carried, notes = F.carried_model(results["select"], results["stability"])
print("\n".join(notes))
print(f"\nCARRIED INTO D5: {carried}")
_ = (TABLES / "decision.json").write_text(json.dumps({
    "carried": carried, "notes": notes,
    "origins": {k: str(v.date()) for k, v in o.__dict__.items()}}, indent=2), encoding="utf-8")

In [ ]:
def pct(x):
    return f"{100 * x:.2f}%"

lines = ["| origin | model | WAPE 24m | WAPE y1 | WAPE y2 | bias 24m | network WAPE | route MAE | route RMSE |",
         "|---|---|---|---|---|---|---|---|---|"]
for (label, name), r in table.iterrows():
    lines.append(f"| {label} | {name} | {pct(r.WAPE_24m)} | {pct(r.WAPE_y1)} | {pct(r.WAPE_y2)} | "
                 f"{100 * r.bias_24m:+.2f}% | {pct(r.network_WAPE)} | {r.route_MAE:,.1f} | {r.route_RMSE:,.1f} |")
md = (f"# Model comparison scores — Team 10 (D4)\n\n_Generated by `notebooks/04_model_comparison.ipynb`._\n\n"
      f"Origins: select {o.select.date()}, stability {o.stability.date()} (24-month one-shot forecasts).\n\n"
      + "\n".join(lines) + "\n\n## Carry rule\n\n" + "\n".join(f"- {n}" for n in notes)
      + f"\n\n**Carried into D5: {carried}**\n")
print("wrote", write_table("model_comparison_scores.md", md))

In [ ]:
# Machine-readable copy for scripts/build_deliverables.py (fills the reports and the slides).
_ = (TABLES / "d4_scores.json").write_text(json.dumps({
    "origins": {k: str(v.date()) for k, v in o.__dict__.items()},
    "carried": carried, "notes": notes,
    "scores": results,
    "by_horizon": {lab: {n: [float(x) for x in s.values] for n, s in v.items()} for lab, v in horizons.items()},
}, indent=2, default=float), encoding="utf-8")
print("wrote", TABLES / "d4_scores.json")